# Agent Probe Notebook

Run each stage (Extraction -> Understanding -> Retrieval -> Analysis -> PPT -> Planner) and inspect outputs for tuning.

This notebook uses `scripts/agent_probe.py` so CLI and notebook results are consistent.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if (ROOT / 'app').exists() is False:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.agent_probe import run_probe

ROOT

In [ ]:
# ---- Tuning Inputs ----
workspace = 'agent-probe-notebook'
inputs_dir = ROOT / 'data' / 'inputs'
request = 'Create an intermediate training deck with practical steps and a summary slide.'
audience = 'intermediate'
top_k = 8
max_blocks = 40
out_dir = ROOT / 'data' / 'agent_probe' / workspace

print('inputs_dir:', inputs_dir)
print('out_dir:', out_dir)

In [ ]:
report = run_probe(
    workspace=workspace,
    inputs_dir=inputs_dir,
    request=request,
    audience=audience,
    top_k=top_k,
    max_blocks=max_blocks,
    out_dir=out_dir,
    force_local=True,
    run_planner=True,
)

{
    'workspace': report['workspace'],
    'llm_available': report['llm_available'],
    'embeddings_available': report['embeddings_available'],
    'blocks': report['extraction']['blocks'],
    'topics': report['understanding']['topic_count'],
    'slides': report['analysis']['metrics']['slides'],
    'deck_bytes': report['ppt']['bytes'],
    'elapsed_seconds': report['elapsed_seconds'],
}

In [ ]:
print('Top topics:', report['understanding']['topics'][:10])
print('Modalities:', report['extraction']['modalities'])

print('\nTop retrieval hits:')
for h in report['indexing']['search_hits'][:5]:
    print(f"- {h['score']:.3f} | {h['source']} | {h['title']}")

In [ ]:
plan = report['analysis']['plan']
print('Deck title:', plan['deck_title'])
print('Audience:', plan['audience'])
print('Slides:', len(plan['slides']))

for i, s in enumerate(plan['slides'][:6], start=1):
    print(f"\n{i}. {s['title']}")
    for b in s['bullets'][:4]:
        print('   -', b)

In [ ]:
# Optional: compare multiple prompt phrasings quickly
requests = [
    'Create a beginner deck focused on key concepts.',
    'Create an executive summary deck with risks and decisions.',
    'Create a hands-on training deck with step-by-step workflow guidance.'
]

comparisons = []
for r in requests:
    rep = run_probe(
        workspace=f"{workspace}-cmp-{len(comparisons)+1}",
        inputs_dir=inputs_dir,
        request=r,
        audience=None,
        top_k=top_k,
        max_blocks=max_blocks,
        out_dir=out_dir / 'comparisons',
        force_local=True,
        run_planner=False,
    )
    comparisons.append({
        'request': r,
        'slides': rep['analysis']['metrics']['slides'],
        'unique_source_block_ids': rep['analysis']['metrics']['unique_source_block_ids'],
        'deck_title': rep['analysis']['deck_title'],
        'topic': rep['analysis']['topic'],
    })

comparisons

In [ ]:
# Inspect saved artifacts
artifact_files = sorted([p.name for p in out_dir.glob('*.json')])
artifact_files

In [ ]:
# Example: load the final report from disk
report_disk = json.loads((out_dir / 'report.json').read_text(encoding='utf-8'))
report_disk.keys()